# Download S3 data to local `data/`

This notebook syncs selected S3 files into the local `data/` folder so you can train and explore without repeated S3 reads.


In [ ]:
from pathlib import Path
import os
import s3fs

# --- Config ---
S3_BUCKET = os.getenv("S3_BUCKET", "zkong-power")
S3_PREFIX = os.getenv("S3_PREFIX", "stack-model/data").strip("/")
LOCAL_DATA_DIR = Path("data").resolve()

# Areas to download (edit this list)
AREAS = ["DE_LU"]

# Core files to sync per area (edit as needed)
AREA_FILES = [
    "day_ahead.csv",
    "day_ahead_real.csv",
    "load_real.csv",
    "load_forecast.csv",
    "load_actual.csv",
]

# Also download weather files (one per area)
INCLUDE_WEATHER = True

# Skip files that already exist locally
SKIP_EXISTING = True

# If True, only prints what would be downloaded
DRY_RUN = False

# Optional: AWS credentials should be present in env vars for s3fs
# AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, AWS_REGION

fs = s3fs.S3FileSystem()

def s3_path(*parts: str) -> str:
    key = "/".join([p.strip("/") for p in parts if p])
    return f"{S3_BUCKET}/{key}"

def download_one(s3_obj: str, local_path: Path) -> bool:
    if SKIP_EXISTING and local_path.exists():
        return False
    if not fs.exists(s3_obj):
        print(f"Missing on S3: {s3_obj}")
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if DRY_RUN:
        print(f"[dry-run] {s3_obj} -> {local_path}")
        return True
    fs.get(s3_obj, str(local_path))
    print(f"Downloaded: {s3_obj} -> {local_path}")
    return True

downloaded = []
for area in AREAS:
    for name in AREA_FILES:
        s3_obj = s3_path(S3_PREFIX, area, name)
        local_path = LOCAL_DATA_DIR / area / name
        if download_one(s3_obj, local_path):
            downloaded.append(local_path)

    if INCLUDE_WEATHER:
        weather_name = f"{area}_weather.csv"
        s3_obj = s3_path(S3_PREFIX, "weather", weather_name)
        local_path = LOCAL_DATA_DIR / "weather" / weather_name
        if download_one(s3_obj, local_path):
            downloaded.append(local_path)

print(f"Done. Downloaded {len(downloaded)} file(s).")


## Quick local check
Run the cell below to confirm files are present locally.


In [ ]:
from pathlib import Path

for area in AREAS:
    area_dir = Path("data") / area
    print(area, list(sorted(p.name for p in area_dir.glob("*"))))
